# 22 · Tablero de EmoGol y monitoreo en Grafana

Este notebook levanta el sistema completo en Google Colab y captura las dos interfaces que ve una persona:

1. **Tablero de EmoGol (Streamlit):** se analiza un video real de YouTube y se captura la pantalla con la distribución, la evolución y los ejemplos.
2. **Grafana:** con el servicio en uso, se captura el tablero de monitoreo de nueve paneles, alimentado por Prometheus.

Las capturas se toman con un navegador sin ventana (Playwright) dentro de Colab, sin exponer ningún puerto a internet.

**Entorno:** Google Colab con GPU T4. **Duración:** unos 20 minutos. **Secretos:** `HF_TOKEN` y `YOUTUBE_API_KEY`.
**Resultado:** capturas en `emotion-futbol-data/prototipo/tablero/`.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes "huggingface_hub<2.0" "pandas<3" joblib fastapi uvicorn httpx redis prometheus_client psutil nvidia-ml-py streamlit playwright
!pip uninstall -y -q torchao
!playwright install --with-deps chromium > /dev/null 2>&1
!apt-get -qq install -y redis-server > /dev/null
!wget -q https://github.com/prometheus/prometheus/releases/download/v2.55.1/prometheus-2.55.1.linux-amd64.tar.gz && tar xzf prometheus-2.55.1.linux-amd64.tar.gz
!wget -q https://dl.grafana.com/oss/release/grafana-11.3.1.linux-amd64.tar.gz && tar xzf grafana-11.3.1.linux-amd64.tar.gz
!rm -rf emotion_futbol && git clone -q https://github.com/santiagortegasegura/emotion_futbol.git
!pip install -q -e "emotion_futbol/emogol_app[servicio]"

In [ ]:
import os, sys, glob, time, subprocess, httpx
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")
login(token=os.environ["HF_TOKEN"])
REPO_MODELO = "santiagortegasegura/emogol-2b"
REPO_DATOS = "santiagortegasegura/emotion-futbol-data"
APP = "/content/emotion_futbol/emogol_app"
sys.path.insert(0, APP)

VIDEO_ID = "6CTMAuxbkNM"   # final de la Copa del Mundo 2026 (estudio de caso del Capítulo 4)
MAXIMO = 600               # comentarios a analizar en el tablero (unos 3 minutos)

def esperar(url, intentos=120, pausa=3):
    for _ in range(intentos):
        try:
            if httpx.get(url, timeout=3).status_code < 500:
                return True
        except Exception:
            pass
        time.sleep(pausa)
    return False

## 1. Levantar Redis, el servicio, Prometheus, Grafana y el tablero

In [ ]:
subprocess.run(["redis-server", "--daemonize", "yes"], check=True)
entorno = {**os.environ, "EMOGOL_JOBLIB": hf_hub_download(REPO_MODELO, "emogol.joblib"),
           "PYTHONPATH": APP, "REDIS_URL": "redis://127.0.0.1:6379/0"}
servidor = subprocess.Popen(["python", "-m", "uvicorn", "api.main:app", "--host", "127.0.0.1", "--port", "8000"],
                            cwd=APP, env=entorno, stdout=open("servidor.log", "w"), stderr=subprocess.STDOUT)

# Prometheus lee /metrics del servicio cada 2 s
open("prometheus.yml", "w").write("global:\n  scrape_interval: 2s\nscrape_configs:\n  - job_name: emogol\n"
                                  "    static_configs:\n      - targets: ['127.0.0.1:8000']\n")
prometheus = subprocess.Popen(["prometheus-2.55.1.linux-amd64/prometheus", "--config.file=prometheus.yml",
                               "--storage.tsdb.path=prom_datos", "--web.listen-address=127.0.0.1:9090"],
                              stdout=open("prometheus.log", "w"), stderr=subprocess.STDOUT)

# Grafana con la misma fuente de datos y el mismo tablero de emogol_app/despliegue, apuntando a 127.0.0.1
os.makedirs("grafana_prov/datasources", exist_ok=True); os.makedirs("grafana_prov/dashboards", exist_ok=True)
open("grafana_prov/datasources/prometheus.yml", "w").write(
    "apiVersion: 1\ndatasources:\n  - name: Prometheus\n    uid: prometheus\n    type: prometheus\n"
    "    access: proxy\n    url: http://127.0.0.1:9090\n    isDefault: true\n")
open("grafana_prov/dashboards/emogol.yml", "w").write(
    "apiVersion: 1\nproviders:\n  - name: EmoGol\n    folder: EmoGol\n    type: file\n"
    f"    options:\n      path: {APP}/despliegue/grafana/dashboards\n")
grafana_dir = [d for d in glob.glob("/content/grafana*") if os.path.isdir(d)][0]
entorno_grafana = {**os.environ, "GF_PATHS_PROVISIONING": "/content/grafana_prov", "GF_PATHS_DATA": "/content/grafana_datos",
                   "GF_SERVER_HTTP_ADDR": "127.0.0.1", "GF_SERVER_HTTP_PORT": "3000",
                   "GF_AUTH_ANONYMOUS_ENABLED": "true", "GF_AUTH_ANONYMOUS_ORG_ROLE": "Viewer",
                   "GF_AUTH_DISABLE_LOGIN_FORM": "true", "GF_USERS_DEFAULT_THEME": "light"}
grafana = subprocess.Popen([f"{grafana_dir}/bin/grafana", "server", f"--homepath={grafana_dir}"],
                           env=entorno_grafana, stdout=open("grafana.log", "w"), stderr=subprocess.STDOUT)

# Tablero de EmoGol
tablero = subprocess.Popen(["streamlit", "run", "tablero/app.py", "--server.port", "8501", "--server.headless", "true",
                            "--browser.gatherUsageStats", "false"],
                           cwd=APP, env={**os.environ, "EMOGOL_API": "http://127.0.0.1:8000"},
                           stdout=open("tablero.log", "w"), stderr=subprocess.STDOUT)

for nombre, url, log in [("Servicio", "http://127.0.0.1:8000/salud", "servidor.log"),
                         ("Prometheus", "http://127.0.0.1:9090/-/ready", "prometheus.log"),
                         ("Grafana", "http://127.0.0.1:3000/api/health", "grafana.log"),
                         ("Tablero", "http://127.0.0.1:8501", "tablero.log")]:
    listo = esperar(url)
    print(f"{nombre}: {'listo' if listo else 'NO RESPONDE'}")
    if not listo:
        print(open(log).read()[-1500:])

## 2. Analizar un video en el tablero y capturarlo

In [ ]:
from playwright.async_api import async_playwright

navegador_pw = await async_playwright().start()
navegador = await navegador_pw.chromium.launch()
pagina = await navegador.new_page(viewport={"width": 1400, "height": 1000}, device_scale_factor=2)
await pagina.goto("http://127.0.0.1:8501", wait_until="networkidle")
await pagina.get_by_label("ID o enlace del video de YouTube").fill(VIDEO_ID)
await pagina.keyboard.press("Enter")
# control deslizante: se deja en el valor de MAXIMO moviendo la flecha desde 3.000 (paso 100)
deslizador = pagina.get_by_role("slider")
await deslizador.focus()
for _ in range((3000 - MAXIMO) // 100):
    await deslizador.press("ArrowLeft")
await pagina.get_by_role("button", name="Analizar").click()
await pagina.get_by_text("Distribución de emociones").wait_for(timeout=20 * 60 * 1000)
await pagina.wait_for_timeout(4000)
await pagina.screenshot(path="tablero_video.png", full_page=True)

await pagina.get_by_role("tab", name="Probar textos").click()
await pagina.get_by_role("button", name="Clasificar").click()
await pagina.wait_for_timeout(6000)
await pagina.screenshot(path="tablero_textos.png", full_page=True)
print("Capturas del tablero listas")

## 3. Generar tráfico y capturar Grafana

Se envían mensajes del chat real durante unos minutos, con niveles de carga distintos, para que los paneles muestren actividad.

In [ ]:
import pandas as pd
from pruebas_carga.generador import nivel

chat = pd.read_csv(hf_hub_download(REPO_DATOS, "prototipo/en_vivo_prueba/en_vivo_prueba_chat_clasificado.csv",
                                   repo_type="dataset"), keep_default_na=False)
textos = chat.sort_values("created_at")["text"].astype(str).tolist()
for tasa in (1.12, 3, 6, 2):
    r = await nivel("http://127.0.0.1:8000", textos, tasa, 60, "grafana", usar_cache=True)
    print(f"{tasa} msg/s -> {r.rendimiento_logrado} msg/s | p95 {r.latencia_p95_ms} ms")
time.sleep(20)

pagina = await navegador.new_page(viewport={"width": 1600, "height": 1100}, device_scale_factor=2)
await pagina.goto("http://127.0.0.1:3000/d/emogol/emogol?orgId=1&from=now-12m&to=now&kiosk", wait_until="networkidle")
await pagina.wait_for_timeout(12000)
await pagina.screenshot(path="grafana_emogol.png", full_page=True)
await navegador.close(); await navegador_pw.stop()
print("Captura de Grafana lista")

## 4. Ver y publicar las capturas

In [ ]:
from IPython.display import Image, display
from huggingface_hub import upload_file

for archivo in ["tablero_video.png", "tablero_textos.png", "grafana_emogol.png"]:
    display(Image(archivo, width=900))
    upload_file(path_or_fileobj=archivo, path_in_repo=f"prototipo/tablero/{archivo}", repo_id=REPO_DATOS, repo_type="dataset")
    print(f"Subido: prototipo/tablero/{archivo}")
for p in (tablero, grafana, prometheus, servidor):
    p.terminate()